In [1]:
import os
import cv2
import numpy as np
import joblib
from datetime import datetime
from mtcnn import MTCNN
from keras_facenet import FaceNet
from sklearn.preprocessing import LabelEncoder, Normalizer, StandardScaler  
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split, GridSearchCV
from ultralytics import YOLO
from collections import deque
import tensorflow as tf
import time
import serial

In [2]:
DATASET_DIR = r"F:\TKH\Semester 1\Embedded System\Final Project\Code\cropped_dataset"
IMG_SIZE = (160, 160)
MARGIN = 20
AUG_PER_IMAGE = 3          
UNKNOWN_THRESHOLD = 0.85   

In [3]:
SVM_PATH = r"F:\TKH\Semester 1\Embedded System\Final Project\Code\svm_model.pkl"
ENCODER_PATH = r"F:\TKH\Semester 1\Embedded System\Final Project\Code\encoder.pkl"
SCALER_PATH = r"F:\TKH\Semester 1\Embedded System\Final Project\Code\scaler.pkl"
RF_PATH = r"F:\TKH\Semester 1\Embedded System\Final Project\Code\rf_model.pkl"
L2_FLAG_PATH = r"F:\TKH\Semester 1\Embedded System\Final Project\Code\l2_normalizer.note"
FACENET_NAME = r"F:\TKH\Semester 1\Embedded System\Final Project\Code\facenet_embedder.note"
YOLO_MODEL_PATH = r"F:\TKH\Semester 1\Embedded System\Final Project\Code\yolov8n.pt"

In [4]:
detector = MTCNN()
embedder = FaceNet()
yolo_model = YOLO("yolov8n.pt") 

Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "c:\Users\hp\anaconda3\Lib\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "c:\Users\hp\anaconda3\Lib\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "c:\Users\hp\anaconda3\Lib\site-packages\lz4\frame\__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.


In [5]:
def square_crop_with_margin(box, img_w, img_h, margin=MARGIN):
    x, y, w, h = box
    x, y = abs(x), abs(y)
    x1, y1 = max(0, x - margin), max(0, y - margin)
    x2, y2 = min(img_w, x + w + margin), min(img_h, y + h + margin)
    bw, bh = x2 - x1, y2 - y1
    side = max(bw, bh)
    cx = x1 + bw // 2
    cy = y1 + bh // 2
    x1 = max(0, cx - side // 2)
    y1 = max(0, cy - side // 2)
    x2 = min(img_w, x1 + side)
    y2 = min(img_h, y1 + side)
    return x1, y1, x2, y2

In [6]:
def align_face(image, box, landmarks=None):
    if landmarks is not None:
        left_eye = np.array(landmarks[0][0])  
        right_eye = np.array(landmarks[1][0])
        angle = np.arctan2(right_eye[1] - left_eye[1], right_eye[0] - left_eye[0]) * 180 / np.pi
        M = cv2.getRotationMatrix2D((image.shape[1]//2, image.shape[0]//2), angle, 1.0)
        aligned = cv2.warpAffine(image, M, (image.shape[1], image.shape[0]))
        x1, y1, x2, y2 = square_crop_with_margin(box, image.shape[1], image.shape[0])
        return aligned[y1:y2, x1:x2]
    else:
        x1, y1, x2, y2 = square_crop_with_margin(box, image.shape[1], image.shape[0])
        return image[y1:y2, x1:x2]

In [7]:
def extract_face(path, required_size=IMG_SIZE):
    img = cv2.imread(path)
    if img is None:
        print(f" Could not read image: {path}")
        return None
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = detector.detect_faces(rgb)
    if not results:
        results_yolo = yolo_model(rgb)
        boxes = results_yolo[0].boxes
        if len(boxes) > 0:
            for box in boxes:
                if int(box.cls) == 0:  # Class 0: 'person'
                    box_coords = box.xyxy[0].cpu().numpy()
                    x1, y1, x2, y2 = map(int, box_coords)
                    face = rgb[y1:y2, x1:x2]
                    h, w = face.shape[:2]
                    center_x, center_y = w // 2, h // 2
                    angle = 0  
                    M = cv2.getRotationMatrix2D((center_x, center_y), angle, 1.0)
                    face = cv2.warpAffine(face, M, (w, h))
                    break
            else:
                print(f" No 'person' detected in fallback: {path}")
                return None
        else:
            print(f" No detection in fallback: {path}")
            return None
    else:
        box = results[0]['box']
        landmarks = [[results[0]['keypoints']['left_eye']], [results[0]['keypoints']['right_eye']]]
        face = align_face(rgb, box, landmarks)
    if face.size == 0:
        return None
    face = cv2.resize(face, required_size)
    face = face.astype("float32") / 255.0
    return face

In [8]:
def augment_face(face_rgb_01):
    aug_samples = []

    flipped = cv2.flip(face_rgb_01, 1)
    aug_samples.append(flipped)
    
    beta = np.random.uniform(-30, 30)
    bgr = cv2.cvtColor((face_rgb_01*255).astype(np.uint8), cv2.COLOR_RGB2BGR)
    bright = cv2.convertScaleAbs(bgr, alpha=1.0, beta=beta)
    bright = cv2.cvtColor(bright, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
    aug_samples.append(bright)

    angle = np.random.uniform(-15, 15)
    M = cv2.getRotationMatrix2D((IMG_SIZE[0]//2, IMG_SIZE[1]//2), angle, 1.0)
    rot = cv2.warpAffine((face_rgb_01*255).astype(np.uint8), M, IMG_SIZE)
    rot = rot.astype(np.float32)/255.0
    aug_samples.append(rot)

    blur = cv2.GaussianBlur((face_rgb_01*255).astype(np.uint8), (3,3), 0)
    blur = blur.astype(np.float32)/255.0
    aug_samples.append(blur)

    shear = np.random.uniform(-0.1, 0.1)
    M = np.array([[1, shear, 0], [0, 1, 0]], dtype=np.float32)
    shear_img = cv2.warpAffine((face_rgb_01*255).astype(np.uint8), M, IMG_SIZE)
    shear_img = shear_img.astype(np.float32)/255.0
    aug_samples.append(shear_img)
    np.random.shuffle(aug_samples)
    return aug_samples[:AUG_PER_IMAGE]

In [9]:
def face_to_embedding(face_rgb_01):
    face_255 = (face_rgb_01*255).astype(np.uint8)
    face_255 = np.expand_dims(face_255, 0)
    embs = embedder.embeddings(face_255)
    return embs[0]

In [10]:
def load_models(svm_path=SVM_PATH, rf_path=RF_PATH, encoder_path=ENCODER_PATH, scaler_path=SCALER_PATH):
    if not (os.path.exists(svm_path) and os.path.exists(rf_path) and os.path.exists(encoder_path) and os.path.exists(scaler_path)):
        print(f" Model files missing: {svm_path}, {rf_path}, {encoder_path}, {scaler_path}")
        return None, None, None, None, None
    try:
        svm = joblib.load(svm_path)
        rf = joblib.load(rf_path)
        encoder = joblib.load(encoder_path)
        scaler = joblib.load(scaler_path)
        l2 = Normalizer(norm="l2")
        print(f"Loaded SVM, RF, encoder, scaler, and L2 normalizer")
        return svm, rf, encoder, scaler, l2
    except Exception as e:
        print(f"Error loading models: {str(e)}")
        return None, None, None, None, None

In [11]:
import pyttsx3
import threading

try:
    voice_engine = pyttsx3.init()
    voice_engine.setProperty('rate', 150)    
    voice_engine.setProperty('volume', 1.0)  
except Exception as e:
    print(f"Voice engine error: {e}")

def speak_access_granted(name, temp=25):
    def run_speech():
        text = f"Hi Mr {name}. The temperature in your car is {temp} degrees."
        try:
            local_engine = pyttsx3.init()
            local_engine.say(text)
            local_engine.runAndWait()
        except:
            pass
        
    t = threading.Thread(target=run_speech)
    t.start()

In [ ]:
def realtime_recognition():
    global arduino  
    try:
        if 'arduino' not in globals() or arduino is None:
            arduino = serial.Serial(port='COM11', baudrate=9600, timeout=0.1)
            time.sleep(2) 
            print(" Arduino Connected")
    except:
        arduino = None
        print("Arduino not connected (Running in simulation mode)")

    last_auth_time = 0
    AUTH_COOLDOWN = 30   

    svm, rf, encoder, scaler, l2 = load_models()
    if svm is None: return

    cap = cv2.VideoCapture(0) 
    win_name = "Real-Time Face Recognition"
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret: break
            
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = detector.detect_faces(rgb)
            
            if results:
                for res in results:
                    box = res['box']
                    landmarks = [[res['keypoints']['left_eye']], [res['keypoints']['right_eye']]]
                    face = align_face(rgb, box, landmarks)
                    if face.size == 0: continue
                    
                    face_resized = cv2.resize(face, IMG_SIZE).astype(np.float32) / 255.0
                    emb = face_to_embedding(face_resized).reshape(1, -1)
                    emb = l2.transform(emb)
                    emb = scaler.transform(emb)
                    
                    probs_svm = svm.predict_proba(emb)[0]
                    probs_rf = rf.predict_proba(emb)[0]
                    probs = (probs_svm + probs_rf) / 2
                    pred_idx = np.argmax(probs)
                    pred_conf = probs[pred_idx]
                    pred_name = encoder.inverse_transform([pred_idx])[0]

                    # --- DECISION LOGIC ---\n
                    if pred_conf >= UNKNOWN_THRESHOLD:
                        display_text = f"{pred_name} ({pred_conf*100:.1f}%)"
                        color = (0, 255, 0)
                        
                        # --- SUCCESS ACTION ---
                        if pred_name != "Unknown":
                            print(f" Opening System for {pred_name}")
                            
                            if arduino:
                                arduino.write(b"SYSTEM_INIT\n")
                            
                            speak_access_granted(pred_name, temp=25)
               
                            print(" Face Verified. Closing Camera and switching to Voice Mode...")
                            cap.release()
                            cv2.destroyAllWindows()
                            return 
                    else:
                        display_text = f"Unknown"
                        color = (0, 0, 255)

                    x1, y1, x2, y2 = square_crop_with_margin(box, rgb.shape[1], rgb.shape[0])
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(frame, display_text, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

            cv2.imshow(win_name, frame)
            if cv2.waitKey(1) & 0xFF == ord('q'): break
            if cv2.getWindowProperty(win_name, cv2.WND_PROP_VISIBLE) < 1: break
                
    finally:
        cap.release()
        cv2.destroyAllWindows()

In [13]:
svm, rf, encoder, scaler, l2 = load_models()
realtime_recognition()

Loaded SVM, RF, encoder, scaler, and L2 normalizer
 Arduino Connected
Loaded SVM, RF, encoder, scaler, and L2 normalizer
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
 Opening System for Abdalah Mohamed
 Face Verified. Closing Camera and switching to Voice Mode...


In [ ]:
import speech_recognition as sr
import serial
import time
import re  

def control_ac_system():
    if 'arduino' not in globals() or arduino is None:
        print(" Error: Arduino connection lost. Run Face ID first.")
        return

    print("\n Voice System Active (Connected to Existing Session)")
    recognizer = sr.Recognizer()
    recognizer.energy_threshold = 3000

    print(" Listening for AC commands... (e.g., 'Set AC to 25')")
    with sr.Microphone() as source:
        recognizer.adjust_for_ambient_noise(source)

        while True:
            try:
                print("Waiting for command...")
     
                source.stream.read(source.CHUNK)
                audio = recognizer.listen(source, timeout=5, phrase_time_limit=5)
                text = recognizer.recognize_google(audio).lower()

                print(f" You said: '{text}'")


                # ----- Hi COMMAND -----
                if "ready" in text or "Hi" in text:
                    print(" System Ready")
                    continue

                # ----- EXIT / THANK YOU -----
                if "thank" in text or "thanks" in text:
                    print(" You're welcome! Exiting Voice Mode.")
                    break

                # ----- REVERSE COMMAND -----
                if "reverse" in text or "turn back" in text or "backwards" in text:
                    print(" Sending Command: REVERSE")
                    if arduino:
                        arduino.write(b"reverse\n")
                    continue

                # ----- AC  COMMANDS -----
                if "ac" in text or "air condition" in text or "temperature" in text:

                    found_numbers = re.findall(r'\d+', text)
                    if found_numbers:
                        temp_value = int(found_numbers[0])
                        valid_temps = [0, 25, 50, 75, 100]
                        if temp_value in valid_temps:
                            print(f" Setting AC to {temp_value}%")

                            if arduino:
                                command_str = f"AC_{temp_value}\n"
                                arduino.write(command_str.encode())
                        else:
                            print(f" {temp_value} is not valid. Use: 0, 25, 50, 75, 100.")

                    else:
                        print(" No temperature number heard.")

            except sr.WaitTimeoutError:
                pass  
            except sr.UnknownValueError:
                pass  
            except Exception as e:
                print(f"Error: {e}")


control_ac_system()


 Voice System Active (Connected to Existing Session)
 Listening for AC commands... (e.g., 'Set AC to 25')
Waiting for command...
 You said: 'set ac 0'
 Setting AC to 0%
Waiting for command...
 You said: 'ac 25'
 Setting AC to 25%
Waiting for command...
Waiting for command...
 You said: '50'
Waiting for command...
 You said: 'ac 50'
 Setting AC to 50%
Waiting for command...
 You said: 'ac 100'
 Setting AC to 100%
Waiting for command...
Waiting for command...
 You said: 'turn back'
 Sending Command: REVERSE
Waiting for command...
Waiting for command...
 You said: 'ac 0'
 Setting AC to 0%
Waiting for command...
 You said: 'thank you'
 You're welcome! Exiting Voice Mode.
